# Exercício Prático — Enclose.horse (Busca)

**Nome:** 
**Data:** 19/04/2026

## Introdução

Neste notebook, implemento o ambiente do jogo enclose.horse e algoritmos de busca para analisar se o cavalo consegue fugir (atingir a borda) e, caso contrário, obter estatísticas de execução dos algoritmos.

## 2) Definição do problema de busca a ser resolvido

- Estado inicial: Posição (i, j) rotulada por C;
- Conjunto de ações: Andar para cima, andar para esquerda, andar para a direita ou andar para baixo;
- Modelo de transição: dado um estado s = (i , j), temos as seguintes transições:
    - T(s, cima) = (i - 1, j);
    - T(s, baixo) = (i + 1, j);
    - T(s, esquerda) = (i, j - 1);
    - T(s, direita) = (i, j + 1);
    <br>
    tal que cada coordenada destino acima não é rotulada por + ou % e não ultrapassa os limites de altura e largura da matriz. 
- Estados objetivo: qualquer estado (i, j) rotulado por vazio, J, M ou A tal que i = 0 ou i = h - 1 ou j = 0 ou j = w - 1, sendo h a altura da matriz e w a largura. Ou seja, qualquer estado que representa uma posição que não é obstáculo e se localiza nas bordas da matriz;
- Custo dos passos: 1;
- Conjunto de todos os estados: todos os estados (i, j) não rotulados por + ou %. Ou seja, todos os estados que representam posições que não são obstáculo. 

## 3) Implementar o ambiente

Nesta seção, implemento o carregamento do arquivo de estado `.txt` e as funções compartilhadas pelos algoritmos de busca (estado inicial, teste de objetivo e função de transição).

In [6]:
from pathlib import Path
from dataclasses import dataclass
from collections import deque


class No:
    def __init__(self, i, j, rotulo):
        self.i = i
        self.j = j
        self.rotulo = rotulo


@dataclass(frozen=True)
class Ambiente:
    largura: int
    altura: int
    matriz: list[list[str]]
    posicao_cavalo: tuple[int, int] | None

    @classmethod
    def from_txt(cls, caminho_arquivo: str | Path) -> "Ambiente":
        caminho = Path(caminho_arquivo)
        linhas = caminho.read_text(encoding="utf-8").splitlines()

        if not linhas:
            raise ValueError("Arquivo vazio.")

        try:
            largura, altura = map(int, linhas[0].split())
        except ValueError as exc:
            raise ValueError("Primeira linha invalida. Use o formato: 'H V' com dois inteiros.") from exc

        if len(linhas) < 1 + altura:
            raise ValueError(
                f"Quantidade de linhas do tabuleiro insuficiente: esperado {altura}, "
                f"recebido {max(0, len(linhas) - 1)}."
            )

        linhas_tabuleiro = linhas[1 : 1 + altura]

        matriz: list[list[str]] = []
        posicao_cavalo: tuple[int, int] | None = None
        for i, linha in enumerate(linhas_tabuleiro):
            if len(linha) != largura:
                raise ValueError(
                    f"Linha {i + 2} com tamanho invalido: esperado {largura}, "
                    f"recebido {len(linha)}."
                )

            row = list(linha)
            matriz.append(row)

            if "C" in row:
                j = row.index("C")
                posicao_cavalo = (i, j)

        return cls(
            largura=largura,
            altura=altura,
            matriz=matriz,
            posicao_cavalo=posicao_cavalo,
        )

    def printar_matriz(self):
        for linha in self.matriz:
            print("".join(linha))

    def estado_inicial(self):
        return self.posicao_cavalo

    def e_estado_final(self, no) -> bool:
        i = no.i
        j = no.j
        rotulo = getattr(no, "rotulo", self.matriz[i][j])
        if rotulo == " " or rotulo == "A" or rotulo == "J" or rotulo == "M":
            if i == 0 or i == self.altura - 1 or j == 0 or j == self.largura - 1:
                return True
        return False

    # Alias para manter a chamada usada nos algoritmos
    def estado_final(self, no) -> bool:
        return self.e_estado_final(no)

    def funcao_transicao(self, no):
        i, j = no.i, no.j
        estados = []
        obstaculos = {"%", "+"}
        if i + 1 < self.altura and self.matriz[i + 1][j] not in obstaculos:  # baixo
            estados.append(No(i + 1, j, self.matriz[i + 1][j]))
        if i - 1 >= 0 and self.matriz[i - 1][j] not in obstaculos:  # cima
            estados.append(No(i - 1, j, self.matriz[i - 1][j]))
        if j + 1 < self.largura and self.matriz[i][j + 1] not in obstaculos:  # direita
            estados.append(No(i, j + 1, self.matriz[i][j + 1]))
        if j - 1 >= 0 and self.matriz[i][j - 1] not in obstaculos:  # esquerda
            estados.append(No(i, j - 1, self.matriz[i][j - 1]))

        return estados

    def calcular_pontuacao(self):
        obstaculos = {"%", "+"}
        fila = deque([self.posicao_cavalo])
        visitados = {self.posicao_cavalo}

        pontuacao = 0

        while fila:
            i, j = fila.popleft()
            celula = self.matriz[i][j]

            # +1 por cada célula alcançável (inclui J/M/A e a posição do cavalo)
            pontuacao += 1
            if celula == "J":
                pontuacao += 3
            elif celula == "M":
                pontuacao += 10
            elif celula == "A":
                pontuacao -= 5

            for di, dj in ((1, 0), (-1, 0), (0, 1), (0, -1)):
                ni, nj = i + di, j + dj
                if 0 <= ni < self.altura and 0 <= nj < self.largura:
                    if (ni, nj) not in visitados and self.matriz[ni][nj] not in obstaculos:
                        visitados.add((ni, nj))
                        fila.append((ni, nj))

        return pontuacao


def carregar_ambiente_txt(caminho_arquivo):
    return Ambiente.from_txt(caminho_arquivo)

In [7]:
# Teste rápido: carregamento e impressão da matriz
ambiente = carregar_ambiente_txt("estados/entice.txt")
ambiente.printar_matriz()
print("Posição do cavalo:", ambiente.posicao_cavalo)

%%%% % %   %%%%
%%     %     %%
%  J   % %    %
%  %   %      %
  % %  %   %%  
 %   % %  %  % 
               
%%%%%% C %%%%%%
               
 %   % %   %%A 
   %   %  %%%% 
% %%%% %      %
%  M%  % %%%  %
%%     %     %%
%%%%   %   %%%%
Posição do cavalo: (7, 7)


## 4) Implementar algoritmos de busca (BFS e DFS)

Nesta seção, implemento BFS e DFS. Ambos retornam também estatísticas de execução: número de nós alcançados (descobertos) e nós expandidos.

In [8]:
def reconstruir_caminho(pais, objetivo_coord):
    caminho = []
    atual = objetivo_coord
    while atual is not None:
        caminho.append(atual)
        atual = pais.get(atual)
    caminho.reverse()
    return caminho


class FronteiraBFS:
    def __init__(self):
        self.fronteira = []

    def adicionar(self, no):
        self.fronteira.append(no)

    def remover(self):
        if not self.fronteira:
            raise ValueError("Fronteira vazia.")
        return self.fronteira.pop(0)

    def esta_vazia(self):
        return len(self.fronteira) == 0


class FronteiraDFS:
    def __init__(self):
        self.fronteira = []

    def adicionar(self, no):
        self.fronteira.append(no)

    def remover(self):
        if not self.fronteira:
            raise ValueError("Fronteira vazia.")
        return self.fronteira.pop()

    def esta_vazia(self):
        return len(self.fronteira) == 0


class Bfs:
    def bfs(matriz, estado_inicial):
        nos_alcancados = set()
        nos_expandidos = 0
        pais = {}

        estado_inicial = No(
            estado_inicial[0],
            estado_inicial[1],
            matriz[estado_inicial[0]][estado_inicial[1]],
        )
        coord_inicial = (estado_inicial.i, estado_inicial.j)
        pais[coord_inicial] = None

        if ambiente.estado_final(estado_inicial):
            return estado_inicial, [coord_inicial], 1, 0

        fronteira = FronteiraBFS()
        fronteira.adicionar(estado_inicial)
        nos_alcancados.add(coord_inicial)
        nos_alcancados_qtd = 1

        while not fronteira.esta_vazia():
            no = fronteira.remover()
            nos_expandidos += 1

            for no_filho in ambiente.funcao_transicao(no):
                coord = (no_filho.i, no_filho.j)
                if coord not in nos_alcancados:
                    pais[coord] = (no.i, no.j)
                    if ambiente.e_estado_final(no_filho):
                        caminho = reconstruir_caminho(pais, coord)
                        return no_filho, caminho, nos_alcancados_qtd + 1, nos_expandidos
                    nos_alcancados.add(coord)
                    nos_alcancados_qtd += 1
                    fronteira.adicionar(no_filho)

        return None, None, nos_alcancados_qtd, nos_expandidos


class Dfs:
    def dfs(matriz, estado_inicial):
        nos_alcancados = set()
        nos_expandidos = 0
        pais = {}

        estado_inicial = No(
            estado_inicial[0],
            estado_inicial[1],
            matriz[estado_inicial[0]][estado_inicial[1]],
        )
        coord_inicial = (estado_inicial.i, estado_inicial.j)
        pais[coord_inicial] = None

        if ambiente.estado_final(estado_inicial):
            return estado_inicial, [coord_inicial], 1, 0

        fronteira = FronteiraDFS()
        fronteira.adicionar(estado_inicial)
        nos_alcancados.add(coord_inicial)
        nos_alcancados_qtd = 1

        while not fronteira.esta_vazia():
            no = fronteira.remover()
            nos_expandidos += 1

            for no_filho in ambiente.funcao_transicao(no):
                coord = (no_filho.i, no_filho.j)
                if coord not in nos_alcancados:
                    pais[coord] = (no.i, no.j)
                    if ambiente.e_estado_final(no_filho):
                        caminho = reconstruir_caminho(pais, coord)
                        return no_filho, caminho, nos_alcancados_qtd + 1, nos_expandidos
                    nos_alcancados.add(coord)
                    nos_alcancados_qtd += 1
                    fronteira.adicionar(no_filho)

        return None, None, nos_alcancados_qtd, nos_expandidos

In [9]:
# Execução dos algoritmos em instâncias disponíveis
for arquivo in [
    "estados/entice.txt",
    "estados/geometry.txt",
    "estados/entice-resolvido.txt",
    "estados/geometry-resolvido.txt",
]:
    ambiente = carregar_ambiente_txt(arquivo)
    matriz = ambiente.matriz
    estado_inicial = ambiente.posicao_cavalo

    print("\nArquivo:", arquivo)
    print("Inicial:", estado_inicial)

    res_bfs, caminho_bfs, alc_bfs, exp_bfs = Bfs.bfs(matriz, estado_inicial)
    print("Objetivo obito pela BFS:", None if res_bfs is None else (res_bfs.i, res_bfs.j))
    print("Tamanho do caminho obtido pela BFS ", None if caminho_bfs is None else len(caminho_bfs) - 1)
    print("Nós alcançados pela BFS:", alc_bfs) 
    print("Nós expandidos pela BFS:", exp_bfs)

    if res_bfs is None:
        print("Sem fuga (BFS). Pontuação do cavalo:", ambiente.calcular_pontuacao())

    res_dfs, caminho_dfs, alc_dfs, exp_dfs = Dfs.dfs(matriz, estado_inicial)
    print("Objetivo obito pela DFS:", None if res_dfs is None else (res_dfs.i, res_dfs.j))
    print("Tamanho do caminho obtido pela DFS:", None if caminho_dfs is None else len(caminho_dfs) - 1)
    print("Nós alcançados pela DFS:", alc_dfs)
    print("Nós expandidos pela DFS:", exp_dfs)

    if res_dfs is None:
        print("Sem fuga (DFS). Pontuação do cavalo:", ambiente.calcular_pontuacao())


Arquivo: estados/entice.txt
Inicial: (7, 7)
Objetivo obito pela BFS: (14, 8)
Tamanho do caminho obtido pela BFS  8
Nós alcançados pela BFS: 76
Nós expandidos pela BFS: 59
Objetivo obito pela DFS: (6, 0)
Tamanho do caminho obtido pela DFS: 8
Nós alcançados pela DFS: 17
Nós expandidos pela DFS: 8

Arquivo: estados/geometry.txt
Inicial: (15, 15)
Objetivo obito pela BFS: (29, 21)
Tamanho do caminho obtido pela BFS  20
Nós alcançados pela BFS: 308
Nós expandidos pela BFS: 278
Objetivo obito pela DFS: (0, 18)
Tamanho do caminho obtido pela DFS: 24
Nós alcançados pela DFS: 70
Nós expandidos pela DFS: 58

Arquivo: estados/entice-resolvido.txt
Inicial: (7, 7)
Objetivo obito pela BFS: None
Tamanho do caminho obtido pela BFS  None
Nós alcançados pela BFS: 56
Nós expandidos pela BFS: 56
Sem fuga (BFS). Pontuação do cavalo: 59
Objetivo obito pela DFS: None
Tamanho do caminho obtido pela DFS: None
Nós alcançados pela DFS: 56
Nós expandidos pela DFS: 56
Sem fuga (DFS). Pontuação do cavalo: 59

Arqui

## 5) Obter resultados

Nesta seção, apresento os resultados de execução (solução encontrada ou não) e as estatísticas (nós alcançados e expandidos) para as instâncias disponibilizadas.

## 6) Discussão

Nesta seção, discuto os resultados e as particularidades de BFS e DFS, incluindo as estatísticas coletadas (nós expandidos, nós descobertos) e os parâmetros do problema (por exemplo, fator de ramificação $b$, profundidade da solução $d$ e profundidade máxima $m$).

*(A implementação de A* será adicionada posteriormente.)